# 実習ノート：動画で上肢の動きを測る

本編の実習①（0〜17）と実習②（18〜21）を、この1本で行います。

- 実習①：動画からMediaPipeで体の33点を出し、肩挙上角を2Dと3Dの両方で計算して、正解（モーションキャプチャー）と比べます。最後に自分の動画も測ります。
- 実習②：手首の3Dの位置から手の速さ（m/s）を出し、速度曲線を正解と比べます。

## このノートの使い方

- 灰色の箱を「セル」と呼びます。セルの中に書いてある文が「コード」（コンピューターへの手順書）です。
- セルの左の▶を押すと、そのセルが実行され、すぐ下に結果が出ます（Shift＋Enterでも同じ）。
- 上のセルから順に、全部押してください。下のセルは、上のセルで作ったものを使います。
- どこかを書き換えたら、そのセルから下を全部押し直してください。
- 結果が出ない・赤い文字が出たときは、メニューの［ランタイム］→［すべてのセルを実行］を押せば、上から順に押し直せます。

## 使うデータ

肩の外転の動画と正解は、REHAB24-6（Masaryk大学、CC BY-NC 4.0）から切り出したものです。患者のデータではありません。

---
# 実習①　動画から肩の角度を測る

## 0. 最初の1行

セルの左の▶を押してみましょう。`print(…)`は、かっこの中を下に表示する命令です。すぐ下に「こんにちは」と出れば成功です。

In [ ]:
print('こんにちは')

## 1. 道具を読み込む

計算やグラフの道具は、最初は箱にしまわれています。`import`は「道具箱を開ける」命令です。
1行目は、動画から体の点を出す道具（MediaPipe）と、グラフに日本語を出す道具を入れる行です（先頭の`!`は、Colabに「これを入れて」と頼む合図）。
1〜2分かかります。赤い文字で`mediapipe`が入らないと出たときは、1行目を`!pip install -q mediapipe matplotlib-fontja`に書き換えて押し直してください。

In [ ]:
!pip install -q mediapipe==0.10.21 matplotlib-fontja
import numpy as np
import matplotlib.pyplot as plt
import matplotlib_fontja
from scipy.signal import find_peaks, savgol_filter
print('道具の準備ができました')

## 2. 動画を持ってくる

`URL = '…'`は、データの置き場所に`URL`という名前を付けて取っておく行です。名前を付けて取っておいたものを「変数」と呼びます。
2行目以降は、その場所から、動画3本（正面・横・斜め）と正解のファイルを持ってくる行です。

In [ ]:
URL = 'https://k0heiun0.github.io/pt_seminar/data/rehab24/'
!wget -q {URL}front.mp4 {URL}side.mp4 {URL}oblique.mp4
!wget -q {URL}front_truth.csv {URL}side_truth.csv {URL}oblique_truth.csv
!wget -q {URL}front_truth_wrist.csv
print('データを持ってきました')

## 3. 道具の関数を作る（中身は読み飛ばしてよい）

動画を扱うための関数を3つ作ります。関数は、入力を受け取って結果を返す手順のまとまりです。中身は読み飛ばしてかまいません。役割・入力・出力だけ読んでください。

| 関数 | 役割 | 入力 | 出力 |
|---|---|---|---|
| `show_frame` | 動画の1コマを表示する | 動画の名前、何秒目か | （画像が出る） |
| `track_pose` | 全コマで体の33点を出す | 動画の名前 | 2Dの点、3Dの点、1秒あたりのコマ数 |
| `draw_points` | 1コマに33点を重ねて表示する | 動画の名前、何秒目か、2Dの点 | （画像が出る） |

1行目は、MediaPipeのモデル（学習済みの重み）を持ってくる行です。このセルを押しても何も表示されません。関数は、作っただけでは動かず、呼んだときに動きます。

In [ ]:
!wget -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task
import cv2
import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision


def read_frame(file, sec):
    video = cv2.VideoCapture(file)
    video.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
    ok, frame = video.read()
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def show_frame(file, sec):
    plt.imshow(read_frame(file, sec))
    plt.axis('off')
    plt.show()


def track_pose(file, max_sec=60):
    # 役割：全コマで体の33点を出す。人が写っていないコマは空欄（nan）にする
    options = vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path='pose_landmarker_full.task'),
        running_mode=vision.RunningMode.VIDEO, num_poses=1)
    video = cv2.VideoCapture(file)
    fps = video.get(cv2.CAP_PROP_FPS)
    points2d, points3d = [], []
    with vision.PoseLandmarker.create_from_options(options) as model:
        while len(points2d) < max_sec * fps:
            ok, frame = video.read()
            if not ok:
                break
            h, w = frame.shape[:2]
            image = mp.Image(image_format=mp.ImageFormat.SRGB,
                             data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            result = model.detect_for_video(image, int(len(points2d) * 1000 / fps))
            if result.pose_landmarks:
                points2d.append([[p.x * w, p.y * h] for p in result.pose_landmarks[0]])
                points3d.append([[p.x, p.y, p.z] for p in result.pose_world_landmarks[0]])
            else:
                points2d.append(np.full((33, 2), np.nan))
                points3d.append(np.full((33, 3), np.nan))
    return np.array(points2d), np.array(points3d), fps


def draw_points(file, sec, points2d, fps):
    k = int(sec * fps)
    plt.imshow(read_frame(file, sec))
    plt.scatter(points2d[k, :, 0], points2d[k, :, 1], s=12, c='yellow')
    for i in [11, 12, 13, 14, 23, 24]:
        plt.annotate(str(i), points2d[k, i], color='white', fontsize=9)
    plt.axis('off')
    plt.show()

## 4. 1コマを見る

作った関数を呼んでみます。`関数の名前(入力)`と書くと、関数が動きます。正面から撮った動画の4秒目を表示します。

In [ ]:
show_frame('front.mp4', 4)

## 5. 体の33点を出す

正面の動画の全コマで、MediaPipeが体の33点を出します。30秒ほどかかります。
返ってくる3つの結果に、`points2d`・`points3d`・`fps`という名前を付けます。

- `points2d`：画像の中の位置（横・縦のピクセル）。2つの数
- `points3d`：腰の中央を原点にした位置（横・上下・奥行き、メートル）。奥行きは推定した値。3つの数

In [ ]:
points2d, points3d, fps = track_pose('front.mp4')
print('コマ数', len(points2d))
print('1秒あたりのコマ数', fps)

## 6. 1コマの点を描く

4秒目のコマに33点を重ねます。数字は点の番号です（11 左肩、12 右肩、13 左肘、14 右肘、23 左股、24 右股）。
`points2d[120][12]`は、120コマ目（4秒目）の12番の点（右肩）です。番号は0から数えます。

In [ ]:
draw_points('front.mp4', 4, points2d, fps)
print('右肩 2D', np.round(points2d[120][12], 1))
print('右肩 3D', np.round(points3d[120][12], 3))

## 7. 2本の線のなす角を出す関数

ゴニオメーターの代わりになる関数です。`def 名前(入力):`で始まり、`return 結果`で終わります。中身は読み飛ばしてかまいません。
役割：2本の線のなす角を出す。入力：2本の線（`v1`・`v2`）。出力：角度（度）。
2本の線が2つの数（2D）でも3つの数（3D）でも、同じ手順で角度が出ます。

In [ ]:
def angle_between_vectors(v1, v2):
    v1 = v1 / np.linalg.norm(v1)
    v2 = v2 / np.linalg.norm(v2)
    cos_angle = np.clip(np.dot(v1, v2), -1.0, 1.0)
    return np.degrees(np.arccos(cos_angle))

## 8. 肩挙上角の関数

ゴニオメーターの当て方をそのまま書きます。軸は右肩、固定アームは体幹の線（両肩の中点→両股の中点）、移動アームは上腕の線（右肩→右肘）です。
役割：1コマの右肩の挙上角を出す。入力：1コマの33点（2Dでも3Dでもよい）。出力：角度（度）。

In [ ]:
def shoulder_elevation(p):
    # 1. どの点を使うか（番号は0から数える）
    shoulder, elbow = p[12], p[14]
    mid_shoulder = (p[11] + p[12]) / 2
    mid_hip = (p[23] + p[24]) / 2
    # 2. 2本の線を作る
    upper_arm = elbow - shoulder        # 移動アーム
    trunk = mid_hip - mid_shoulder      # 固定アーム
    # 3. なす角を返す
    return angle_between_vectors(upper_arm, trunk)

## 9. 1コマだけ測る

同じ関数に、2Dの点と3Dの点を渡します。`round(…, 1)`は小数第1位までに丸める命令です。

In [ ]:
print('4秒目の肩挙上角 2D', round(shoulder_elevation(points2d[120]), 1))
print('4秒目の肩挙上角 3D', round(shoulder_elevation(points3d[120]), 1))

## 10. 全コマを測る

同じ測定を、全部のコマで繰り返します。`for k in range(len(points2d)):`は「`k`を0、1、2…と変えながら、下の字下げした行を繰り返す」という意味です。
測った角度は、空の入れ物（`[]`）に1つずつ足していきます（`append`）。

In [ ]:
angle2d = []
angle3d = []
for k in range(len(points2d)):
    angle2d.append(shoulder_elevation(points2d[k]))
    angle3d.append(shoulder_elevation(points3d[k]))
angle2d = np.array(angle2d)
angle3d = np.array(angle3d)
print('2Dの最大', round(np.nanmax(angle2d), 1))
print('3Dの最大', round(np.nanmax(angle3d), 1))

## 11. 正解を読み込む

正解のファイルから、3列目（肩挙上角）だけを読み込みます。`skiprows=1`は1行目（列の名前）を飛ばす、`usecols=2`は3列目だけ使う、という意味です（番号は0から数える）。

In [ ]:
truth = np.loadtxt('front_truth.csv', delimiter=',', skiprows=1, usecols=2)
print('正解の最大', round(truth.max(), 1))

## 12. 時系列のグラフを描く

横軸を時間、縦軸を肩挙上角にして、正解・2D・3Dを重ねて描きます。
`plt.plot(横, 縦, label=名前)`が線を引く命令、`plt.legend()`が凡例、`plt.show()`が表示です。

In [ ]:
time = np.arange(len(angle2d)) / fps
plt.plot(time, truth, label='正解', color='black')
plt.plot(time, angle2d, label='2D')
plt.plot(time, angle3d, label='3D')
plt.xlabel('時間 [秒]')
plt.ylabel('肩挙上角 [度]')
plt.legend()
plt.show()

## 13. 正解との差を数にする

コマごとの差の大きさ（絶対値）を平均します。`np.nanmean`は、空欄（nan）を飛ばして平均する命令です。

In [ ]:
print('2Dと正解の差', round(np.nanmean(np.abs(angle2d - truth)), 1))
print('3Dと正解の差', round(np.nanmean(np.abs(angle3d - truth)), 1))

## 14. ピークのばらつきとMDC

挙上の回ごとのピークを数え、平均と標準偏差（SD）、最小可検変化量 $\mathrm{MDC}_{95} = 1.96 \times \sqrt{2} \times \mathrm{SEM}$ を出します（SEMの目安に、同じ人の繰り返しのSDを使う）。
正解・2D・3Dの3つで同じことをしたいので、関数にまとめます。

In [ ]:
def peak_summary(angle):
    angle = np.nan_to_num(angle)     # 空欄は0として山を探す
    peaks, _ = find_peaks(angle, height=90, prominence=30)
    values = angle[peaks]
    sd = values.std(ddof=1)
    return len(peaks), values.mean(), sd, 1.96 * np.sqrt(2) * sd

for name, angle in [('正解', truth), ('2D', angle2d), ('3D', angle3d)]:
    n, mean, sd, mdc = peak_summary(angle)
    print(name, '回数', n, '平均', round(mean, 1), 'SD', round(sd, 1),
          'MDC95', round(mdc, 1))

## 15. 撮る向きを変える

同じ人の外転を、横から撮った動画（`side.mp4`）と、体を斜めに向けた動画（`oblique.mp4`）でも測り、正解との差を比べます。
5・10・11・13のセルでしたことを、関数にまとめて3本に当てます。3本で1〜2分かかります。

In [ ]:
def compare_with_truth(name):
    p2, p3, f = track_pose(name + '.mp4')
    a2 = np.array([shoulder_elevation(p) for p in p2])
    a3 = np.array([shoulder_elevation(p) for p in p3])
    t = np.loadtxt(name + '_truth.csv', delimiter=',', skiprows=1, usecols=2)
    return np.nanmean(np.abs(a2 - t)), np.nanmean(np.abs(a3 - t))

for name in ['front', 'oblique', 'side']:
    d2, d3 = compare_with_truth(name)
    print(name, '2Dの差', round(d2, 1), '3Dの差', round(d3, 1))

## 16. 自分の動画を測る

自分で撮った動画で、同じことをします（正解はありません）。腕を横に上げる動きを、正面から全身が入るように撮ってください。
セルを押すと「ファイルを選択」が出るので、動画を選びます。長い動画は、最初の60秒だけ測ります。

In [ ]:
from google.colab import files
uploaded = files.upload()
my_file = list(uploaded.keys())[0]
my2d, my3d, my_fps = track_pose(my_file)
my_angle2d = np.array([shoulder_elevation(p) for p in my2d])
my_angle3d = np.array([shoulder_elevation(p) for p in my3d])
my_time = np.arange(len(my2d)) / my_fps
plt.plot(my_time, my_angle2d, label='2D')
plt.plot(my_time, my_angle3d, label='3D')
plt.xlabel('時間 [秒]')
plt.ylabel('肩挙上角 [度]')
plt.legend()
plt.show()

## 17. 実習①の問い

グラフと数を見ながら、気づいたことをメモしてください。

1. 正面の動画で、2Dと3Dのどちらが正解に近いか。ピークはどちらに外れているか。
2. 2D・3D・正解で、ピークのSDとMDCはどれくらい違うか。
3. 横から撮ると、なぜ外転が測れないのか（動画の1コマを見て考える）。
4. 斜めから撮ったとき、3Dは2Dより正解に近づいたか。
5. 自分の動画を「良くなった」の比較に使うなら、撮り方の何をそろえるか。

---
# 実習②　手の動きの質を見る

## 18. 手首の速さを出す

5のセルで出した正面の動画の3Dの点（メートル）を使います。16番の点が右手首です。腰の中央からの位置にしてから、隣どうしのコマで動いた距離を出します。
速さ＝1コマで動いた距離×1秒あたりのコマ数、です。

In [ ]:
wrist = points3d[:, 16] - (points3d[:, 23] + points3d[:, 24]) / 2
step = np.linalg.norm(np.diff(wrist, axis=0), axis=1)
speed = step * fps
print('速さの最大（m/s）', round(np.nanmax(speed), 2))

## 19. なめらかにして正解と重ねる

生の速さは、コマごとにぎざぎざです。推定した位置が、コマごとに少しずつ揺れるためです。`savgol_filter`で前後15コマを使ってなめらかにします。
正解の手首の位置（モーションキャプチャー）からも、同じ手順で速さを出して重ねます。

In [ ]:
truth_wrist = np.loadtxt('front_truth_wrist.csv', delimiter=',', skiprows=1,
                         usecols=(2, 3, 4))
truth_speed = np.linalg.norm(np.diff(truth_wrist, axis=0), axis=1) * fps
smooth = savgol_filter(np.nan_to_num(speed), window_length=15, polyorder=2)
truth_smooth = savgol_filter(truth_speed, window_length=15, polyorder=2)
t = np.arange(len(speed)) / fps
plt.plot(t, speed, color='lightgray', label='MediaPipe（生）')
plt.plot(t, smooth, label='MediaPipe（なめらか）')
plt.plot(t, truth_smooth, color='black', label='正解')
plt.xlabel('時間 [秒]')
plt.ylabel('手首の速さ [m/s]')
plt.legend()
plt.show()

## 20. 速さの山を数える

12のセルの挙上のピークと同じ道具（`find_peaks`）で、速さの山を数えます。いちばん速い山の10%より低い山は数えません。
腕を1回上げて下ろすと、上げるときと下ろすときで、山が2つできます。

In [ ]:
for name, s in [('MediaPipe', smooth), ('正解', truth_smooth)]:
    peaks, _ = find_peaks(s, height=s.max() * 0.1, prominence=s.max() * 0.1)
    print(name, '山の数', len(peaks), 'いちばん速いとき', round(s.max(), 2))
print('MediaPipeと正解の相関', round(np.corrcoef(smooth, truth_smooth)[0, 1], 2))

## 21. 実習②の問い

1. 速度曲線は、上げる・下ろすの1回ごとに、山が1つずつ（ベル型）になっているか。
2. MediaPipeの速さは、正解とどれくらい重なっているか。どこで外れているか。
3. 動きの滑らかさを「良くなった」の比較に使うなら、何をそろえて撮ればよいか。